# N3 – Modelagem Preditiva, Avaliação e Rastreio de Experimentos
## Case: Previsão do Preço de Fechamento do Bitcoin (BTC-USD)

**Grupo:** Ana Carolina Fanhani Stralioti, Jairson Steinert, Gustavo do Prado  
**Período dos dados:** 2021-01-01 → 2026-03-19  
**Fonte:** Yahoo Finance via `yfinance`  
**Modelos:** ARIMA (Seminário 2 — AR/MA/ARIMA), Holt-Winters (Seminário 4) e Prophet (Seminário 5)  

---

### Descrição do Case

O Bitcoin (BTC) é a maior criptomoeda do mundo por capitalização de mercado (~US$ 1,5 tri em 2025).  
Seu mercado opera **24 horas por dia, 7 dias por semana**, sem interrupções de final de semana,  
tornando-o uma série temporal diária de alta volatilidade e sem sazonalidade clássica.

**Problema de negócio:** Prever o preço de fechamento para os próximos 7–30 dias,  
auxiliando investidores no timing de posições, gestores de risco na calibração de  
exposição e exchanges na estimação de liquidez futura.

**Conexão com os seminários:**
- **Modelo 1 — ARIMA** (Seminário 2): série não-estacionária, d=1, ACF/PACF guiam p e q.
- **Modelo 2 — Holt-Winters** (Seminário 4): suavização com tendência e sazonalidade semanal.
- **Modelo 3 — Prophet** (Seminário 5): estrutura aditiva/multiplicativa com sazonalidades múltiplas.


In [ ]:
# Instalação das dependências (execute apenas se necessário)
# !pip install yfinance statsmodels prophet wandb scikit-learn matplotlib seaborn scipy -q


In [ ]:
import os
os.makedirs('figuras', exist_ok=True)
FIGS = 'figuras/'

import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Carrega variáveis do .env (se existir) — chave nunca fica no código
try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
    load_dotenv()  # fallback: procura .env no diretório atual
except ImportError:
    pass  # sem dotenv, usa variáveis de ambiente já exportadas

try:
    from prophet import Prophet
    PROPHET_OK = True
    print('Prophet: OK')
except ImportError:
    PROPHET_OK = False
    print('Prophet nao instalado. Execute: pip install prophet')

try:
    import wandb
    _wandb_key = os.environ.get('WANDB_API_KEY', '')
    if _wandb_key:
        wandb.login(key=_wandb_key, relogin=True)
        WANDB_OK = True
        print('wandb: OK (autenticado via .env)')
    else:
        print('wandb: WANDB_API_KEY nao encontrada. Configure o arquivo .env')
        WANDB_OK = False
except ImportError:
    WANDB_OK = False
    print('wandb nao instalado. Execute: pip install wandb')

plt.rcParams['figure.figsize'] = (14, 6)
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')

print('\nImports concluidos!')


---
## Etapa 1 – Revisão e Auditoria da Série Temporal


In [ ]:
# Download dos dados brutos (preços em USD, sem normalização)
ticker = 'BTC-USD'
df_raw = yf.download(ticker, start='2021-01-01', end='2026-03-20',
                     auto_adjust=True, progress=False)

if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df_raw.index = pd.to_datetime(df_raw.index)
df_raw.index.name = 'Date'
df = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

print('=== 1. CONFERENCIA ESTRUTURAL ===')
print(f'Dimensao: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Periodo:  {df.index.min().date()} -> {df.index.max().date()}')
freq_inf = pd.infer_freq(df.index)
print(f'Frequencia inferida: {freq_inf if freq_inf else "Irregular (gaps detectados)"}')

print('\n=== 2. QUALIDADE DO INDICE TEMPORAL ===')
print('Nulos por coluna:')
print(df.isnull().sum())
print(f'\nDatas duplicadas: {df.index.duplicated().sum()}')

idx_completo = pd.date_range(df.index.min(), df.index.max(), freq='D')
gaps = idx_completo.difference(df.index)
print(f'\nTotal datas no calendario: {len(idx_completo)}')
print(f'Datas no dataset:         {len(df)}')
print(f'Gaps detectados:          {len(gaps)} datas ausentes')
if len(gaps) > 0:
    print('Exemplos de gaps (primeiros 5):')
    for g in gaps[:5]:
        print(f'  {g.date()} ({g.day_name()})')
    print('-> Gaps sao finais de semana/feriados: normal para dados do Yahoo Finance.')

df.head(3)


In [ ]:
print('=== 3. DETECCAO DE OUTLIERS (IQR 1.5x) ===')

Q1  = df['Close'].quantile(0.25)
Q3  = df['Close'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df[(df['Close'] < lim_inf) | (df['Close'] > lim_sup)]
print(f'Q1={Q1:,.0f} | Q3={Q3:,.0f} | IQR={IQR:,.0f}')
print(f'Limite inferior: {lim_inf:,.0f}')
print(f'Limite superior: {lim_sup:,.0f}')
print(f'Outliers detectados: {len(outliers)} dias')
if len(outliers) > 0:
    print(outliers[['Close']].head())

print('\n=== 4. IMPUTACAO ===')
print('Nulos detectados:', df.isnull().sum().sum())
print('Acao: nenhuma imputacao necessaria (dados sem nulos).')

# Tabela de diagnostico de qualidade
diag = pd.DataFrame({
    'Verificacao': ['Nulos (total)', 'Datas duplicadas', 'Gaps de calendario', 'Outliers Close (IQR 1.5x)'],
    'Quantidade':  [df.isnull().sum().sum(), df.index.duplicated().sum(), len(gaps), len(outliers)],
    'Acao':        ['Nenhuma', 'Nenhuma', 'Manter (fins de semana)', 'Manter (eventos reais de mercado)']
})
print('\nTABELA - DIAGNOSTICO DE QUALIDADE DE DADOS')
print(diag.to_string(index=False))

print('\n--- Impacto na modelagem ---')
print('Os gaps de calendario (finais de semana) nao impactam a modelagem porque o Bitcoin')
print('opera 24/7 e o Yahoo Finance fornece dados diarios para todos os dias com negociacao.')
print('Os outliers (picos de preco em 2021 e 2024) sao preservados pois representam')
print('eventos reais de mercado (bull runs). Removê-los distorceria a serie historica.')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(df.index, df['Close'], color='#2c7bb6', linewidth=1.2, label='Preco de Fechamento (USD)')
axes[0].set_title('Bitcoin BTC-USD - Preco de Fechamento Diario (2021-2026)', fontsize=14)
axes[0].set_ylabel('Preco (USD)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

log_ret = np.log(df['Close'] / df['Close'].shift(1)).dropna()
axes[1].plot(log_ret.index, log_ret, color='#d7191c', linewidth=0.8, alpha=0.8,
             label='Retorno Logaritmico Diario')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Retornos Logaritmicos Diarios do Bitcoin', fontsize=14)
axes[1].set_ylabel('Log-Retorno')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(f'{FIGS}fig_01_serie_historica.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Etapa 2 – Engenharia de Atributos


In [ ]:
# Funcao auxiliar de metricas (definida aqui para uso em todo o notebook)
def metricas(y_true, y_pred, nome):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {'Modelo': nome, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE_%': round(mape, 2)}

resultados = []  # acumula metricas de todos os modelos

# --- Engenharia de atributos ---
df_feat = df.copy()

# Lags autorregressivos do preco
df_feat['lag1'] = df_feat['Close'].shift(1)
df_feat['lag2'] = df_feat['Close'].shift(2)
df_feat['lag7'] = df_feat['Close'].shift(7)

# Retornos logaritmicos e seus lags
df_feat['log_return']   = np.log(df_feat['Close'] / df_feat['Close'].shift(1))
df_feat['lag1_return']  = df_feat['log_return'].shift(1)
df_feat['lag2_return']  = df_feat['log_return'].shift(2)

# Medias moveis (tendencia de curto e medio prazo)
df_feat['mm7']  = df_feat['Close'].rolling(window=7).mean()
df_feat['mm30'] = df_feat['Close'].rolling(window=30).mean()

# Volatilidade realizada (risco recente)
df_feat['vol_7'] = df_feat['Close'].rolling(window=7).std()

# Encoding ciclico (respeita natureza periodica de mes e dia da semana)
df_feat['mes_sin']     = np.sin(2 * np.pi * df_feat.index.month / 12)
df_feat['mes_cos']     = np.cos(2 * np.pi * df_feat.index.month / 12)
df_feat['dia_sem_sin'] = np.sin(2 * np.pi * df_feat.index.dayofweek / 7)
df_feat['dia_sem_cos'] = np.cos(2 * np.pi * df_feat.index.dayofweek / 7)

# Remove linhas com NaN geradas pelas janelas de rolling/shift
df_feat = df_feat.dropna()
print(f'Dataset com features: {df_feat.shape[0]} obs x {df_feat.shape[1]} colunas')
df_feat.head(3)


In [ ]:
hipoteses = pd.DataFrame({
    'Feature':   ['lag1', 'lag2', 'lag7', 'lag1_return', 'lag2_return',
                  'mm7', 'mm30', 'vol_7', 'mes_sin/mes_cos', 'dia_sem_sin/dia_sem_cos'],
    'Hipotese':  [
        'Preco de ontem tem forte correlacao com hoje (processo AR(1))',
        'Captura dependencia de 2 periodos atras',
        'Captura efeito semanal (ciclos de 7 dias no cripto)',
        'Retorno de ontem pode prever o retorno de hoje (autocorrelacao MA)',
        'Retorno de 2 dias atras como sinal adicional',
        'Tendencia de curto prazo (suaviza ruido diario)',
        'Tendencia de medio prazo / sentimento do mercado',
        'Alta volatilidade recente indica regime de risco elevado',
        'Sazonalidade anual: historico mostra rallies em Q4 (outubro-dezembro)',
        'Sazonalidade semanal: padroes de segunda a domingo'
    ]
})
print('TABELA DE HIPOTESES DAS FEATURES')
print(hipoteses.to_string(index=False))


In [ ]:
# Separacao treino/teste sem embaralhar (80% / 20%)
N = len(df_feat)
split_idx = int(N * 0.8)
train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

# Series de Close (sem normalizacao - correcao do data leakage original)
y_train = train['Close']
y_test  = test['Close']

print(f'Treino: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} obs, {len(train)/N:.1%})')
print(f'Teste:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} obs, {len(test)/N:.1%})')
print('CORRECAO DO DATA LEAKAGE ORIGINAL:')
print('  No notebook anterior, o MinMaxScaler foi ajustado em TODOS os dados antes do split.')
print('  Neste notebook, trabalhamos com precos brutos (USD) para os modelos de series temporais.')
print('  O scaler, quando necessario, sera ajustado APENAS no conjunto de treino.')

plt.figure(figsize=(14, 5))
plt.plot(y_train, label='Treino', color='#2c7bb6', linewidth=1.2)
plt.plot(y_test,  label='Teste',  color='#d7191c', linewidth=1.2)
plt.axvline(x=train.index[-1], color='gray', linestyle='--', alpha=0.7, label='Corte treino/teste')
plt.title('Split Treino/Teste - Preco de Fechamento do Bitcoin (BTC-USD)')
plt.ylabel('Preco (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIGS}fig_02_split.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Etapa 3 – Baselines e Régua de Desempenho


In [ ]:
# Baseline 1: Persistencia (Naive) - preco de ontem como previsao de hoje
y_naive = test['lag1'].values
m_naive = metricas(y_test.values, y_naive, 'Baseline Naive (Persistencia)')
resultados.append(m_naive)

# Baseline 2: Media Movel 7 dias
y_mm7 = test['mm7'].values
m_mm7 = metricas(y_test.values, y_mm7, 'Baseline MM7')
resultados.append(m_mm7)

df_base = pd.DataFrame([m_naive, m_mm7])
print('REGUA DE DESEMPENHO - BASELINES')
print(df_base.to_string(index=False))

plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values,  'k-',   linewidth=1.8, label='Real',  alpha=0.9)
plt.plot(y_test.index, y_naive,        'r--',  linewidth=1.2, label='Naive', alpha=0.8)
plt.plot(y_test.index, y_mm7,          'b--',  linewidth=1.2, label='MM7',   alpha=0.8)
plt.title('Baselines vs. Preco Real (Conjunto de Teste)')
plt.ylabel('Preco BTC (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIGS}fig_03_baselines.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nMeta a superar: RMSE < {m_naive["RMSE"]:,.0f} (Naive) e MAE < {m_naive["MAE"]:,.0f}')


---
## Etapa 4 – Estacionariedade, ACF/PACF e Evidências


In [ ]:
# Decomposicao da serie (periodo=30 dias para capturar ciclo mensal)
serie_close = df['Close'].copy()
decomp = seasonal_decompose(serie_close, model='multiplicative', period=30, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 14))
axes[0].plot(decomp.observed,  color='#2c7bb6'); axes[0].set_title('Serie Original (Close USD)',  fontsize=12)
axes[1].plot(decomp.trend,     color='#e41a1c'); axes[1].set_title('Tendencia',                   fontsize=12)
axes[2].plot(decomp.seasonal,  color='#4daf4a'); axes[2].set_title('Sazonalidade (periodo=30)',    fontsize=12)
axes[3].plot(decomp.resid,     color='#984ea3'); axes[3].set_title('Residuo',                     fontsize=12)
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGS}fig_04_decomposicao.png', dpi=120, bbox_inches='tight')
plt.show()

print('Observacoes da decomposicao:')
print('  Tendencia: forte crescimento de 2021 a 2021-11, queda em 2022 (crash FTX), recuperacao 2023-2025.')
print('  Sazonalidade (30 dias): componente fraca, indicando que BTC nao tem sazonalidade mensal clara.')
print('  Residuo: clusters de volatilidade visiveis - caracteristica de series financeiras (ARCH).')


In [ ]:
def teste_adf(serie, nome):
    res = adfuller(serie.dropna(), autolag='AIC')
    print(f'\nTESTE ADF: {nome}')
    print(f'  Estatistica ADF : {res[0]:.4f}')
    print(f'  p-value          : {res[1]:.6f}')
    print(f'  Lags usados      : {res[2]}')
    criticos = res[4]
    print(f'  Valores criticos : 1%={criticos["1%"]:.3f} | 5%={criticos["5%"]:.3f} | 10%={criticos["10%"]:.3f}')
    if res[1] <= 0.05:
        print('  CONCLUSAO: ESTACIONARIA (rejeita H0, p <= 0.05)')
    else:
        print('  CONCLUSAO: NAO ESTACIONARIA (nao rejeita H0, p > 0.05)')
    return res[1]

p_nivel = teste_adf(serie_close, 'Close - Serie em Nivel')

serie_diff = serie_close.diff().dropna()
p_diff     = teste_adf(serie_diff, 'Delta_Close - 1a Diferenca')

print('\nCONCLUSAO SOBRE DIFERENCIACAO:')
print(f'  Serie em nivel:   p={p_nivel:.4f} -> NAO ESTACIONARIA -> precisamos d=1')
print(f'  1a diferenca:     p={p_diff:.6f} -> ESTACIONARIA -> d=1 suficiente para ARIMA')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_acf(serie_close,  ax=axes[0, 0], lags=50,
         title='ACF - Serie Original (Close)', color='steelblue')
plot_pacf(serie_close, ax=axes[0, 1], lags=50,
          title='PACF - Serie Original (Close)', color='steelblue', method='ywm')
plot_acf(serie_diff,   ax=axes[1, 0], lags=50,
         title='ACF - 1a Diferenca (Delta Close)', color='coral')
plot_pacf(serie_diff,  ax=axes[1, 1], lags=50,
          title='PACF - 1a Diferenca (Delta Close)', color='coral', method='ywm')

plt.tight_layout()
plt.savefig(f'{FIGS}fig_05_acf_pacf.png', dpi=120, bbox_inches='tight')
plt.show()

print('INTERPRETACAO ACF/PACF (serie diferenciada):')
print('  ACF:  decaimento rapido apos lag 1-2 -> componente MA baixa (q=0 ou 1)')
print('  PACF: corte abrupto apos lag 1-2 -> componente AR baixa (p=0 ou 1)')
print('  Sugestao inicial de ordem: ARIMA(1,1,1) ou ARIMA(0,1,1) ou ARIMA(1,1,0)')
print('  Confirmacao via AIC na Etapa 6.')

# Tabela de evidencias EDA
evid = pd.DataFrame({
    'Evidencia':              ['Serie no tempo', 'Decomposicao', 'ADF (nivel)', 'ADF (1a diff)', 'ACF/PACF'],
    'Resultado/Observacao':   [
        'Tendencia crescente, alta volatilidade, sem sazonalidade anual clara',
        'Tendencia dominante; sazonalidade fraca (30 dias); residuos com clusters de volatilidade',
        f'p={p_nivel:.4f} -> NAO ESTACIONARIA (d=1 necessario)',
        f'p={p_diff:.6f} -> ESTACIONARIA (d=1 suficiente)',
        'ACF: corte em lag 1-2 (MA baixo); PACF: corte em lag 1-2 (AR baixo) -> ARIMA(p<=2, 1, q<=2)'
    ]
})
print('\nTABELA DE EVIDENCIAS EDA')
print(evid.to_string(index=False))


---
## Etapa 5 – Modelagem Preditiva (mínimo 3 modelos)

| Modelo | Família | Seminário |
|--------|---------|----------|
| ARIMA(p,1,q) | Modelos lineares AR/MA | Seminário 2 |
| Holt-Winters (aditivo, p=7) | Suavização exponencial | Seminário 4 |
| Prophet (multiplicativo) | Modelo moderno / ML | Seminário 5 |


In [ ]:
print('=== MODELO 1: ARIMA(1,1,1) ===')
print('Justificativa: serie nao-estacionaria (d=1), ACF/PACF sugerem p=1, q=1.')
print('Treinamento no conjunto de treino (precos brutos USD, sem normalizacao).')

order_arima = (1, 1, 1)
fit_arima   = ARIMA(y_train, order=order_arima).fit()
print(fit_arima.summary())

# Previsao out-of-sample para o conjunto de teste
fc_arima       = fit_arima.forecast(steps=len(y_test))
fc_arima.index = y_test.index

m_arima = metricas(y_test.values, fc_arima.values, f'ARIMA{order_arima}')
resultados.append(m_arima)
print(f'\nMetricas ARIMA(1,1,1): MAE={m_arima["MAE"]:,.0f} | RMSE={m_arima["RMSE"]:,.0f} | MAPE={m_arima["MAPE_%"]:.2f}%')


In [ ]:
print('=== MODELO 2: HOLT-WINTERS (Suavizacao Exponencial) ===')
print('Justificativa: serie tem tendencia clara. Usamos Holt-Winters Aditivo')
print('com sazonalidade semanal (period=7), pois Bitcoin pode ter ciclos de 7 dias.')

fit_hw = ExponentialSmoothing(
    y_train,
    trend='add',
    seasonal='add',
    seasonal_periods=7,
    initialization_method='estimated'
).fit(optimized=True)

fc_hw       = fit_hw.forecast(steps=len(y_test))
fc_hw.index = y_test.index

m_hw = metricas(y_test.values, fc_hw.values, 'Holt-Winters (adit, p=7)')
resultados.append(m_hw)
print(f'Parametros: alpha={fit_hw.params["smoothing_level"]:.4f}, ')
print(f'            beta={fit_hw.params["smoothing_trend"]:.4f}, ')
print(f'            gamma={fit_hw.params["smoothing_seasonal"]:.4f}')
print(f'Metricas HW: MAE={m_hw["MAE"]:,.0f} | RMSE={m_hw["RMSE"]:,.0f} | MAPE={m_hw["MAPE_%"]:.2f}%')


In [ ]:
if PROPHET_OK:
    print('=== MODELO 3: PROPHET (Multiplicativo) ===')
    print('Justificativa: modelo moderno que decompose tendencia, sazonalidade anual')
    print('e semanal automaticamente. Modo multiplicativo adequado para series com')
    print('amplitude crescente (como o BTC).')

    df_p_train = pd.DataFrame({'ds': y_train.index, 'y': y_train.values}).reset_index(drop=True)

    model_p = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative'
    )
    model_p.fit(df_p_train)

    # Previsao diretamente para as datas do teste
    df_future = pd.DataFrame({'ds': y_test.index})
    fc_p_df   = model_p.predict(df_future)
    fc_prophet = pd.Series(fc_p_df['yhat'].values, index=y_test.index)

    m_prophet = metricas(y_test.values, fc_prophet.values, 'Prophet (multiplicativo)')
    resultados.append(m_prophet)
    print(f'Metricas Prophet: MAE={m_prophet["MAE"]:,.0f} | RMSE={m_prophet["RMSE"]:,.0f} | MAPE={m_prophet["MAPE_%"]:.2f}%')

    # Componentes do Prophet
    df_future_full = model_p.make_future_dataframe(periods=len(y_test))
    fc_full = model_p.predict(df_future_full)
    fig_comp = model_p.plot_components(fc_full)
    plt.suptitle('Componentes do Modelo Prophet - Bitcoin', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGS}fig_06_prophet_componentes.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Prophet nao disponivel. Instale com: pip install prophet')
    print('Substituindo por SARIMA como Modelo 3...')

    fit_sarima  = ARIMA(y_train, order=(1, 1, 1), seasonal_order=(1, 0, 1, 7)).fit()
    fc_sarima   = fit_sarima.forecast(steps=len(y_test))
    fc_sarima.index = y_test.index
    fc_prophet  = fc_sarima
    m_prophet   = metricas(y_test.values, fc_sarima.values, 'SARIMA(1,1,1)(1,0,1,7)')
    resultados.append(m_prophet)
    print(f'Metricas SARIMA: MAE={m_prophet["MAE"]:,.0f} | RMSE={m_prophet["RMSE"]:,.0f} | MAPE={m_prophet["MAPE_%"]:.2f}%')


---
## Etapa 6 – Hiperparâmetros, AIC e Overfitting


In [ ]:
print('SELECAO DE ORDEM DO ARIMA POR AIC')
print('Testando ARIMA(p, d=1, q) com p in [0..3] e q in [0..3]...')

aic_tab = []
for p in range(0, 4):
    for q in range(0, 4):
        try:
            res = ARIMA(y_train, order=(p, 1, q)).fit()
            aic_tab.append({'p': p, 'd': 1, 'q': q, 'AIC': round(res.aic, 2), 'BIC': round(res.bic, 2)})
        except:
            pass

df_aic = pd.DataFrame(aic_tab).sort_values('AIC').reset_index(drop=True)
df_aic.index += 1
print(df_aic.head(10).to_string())

best_p = int(df_aic.iloc[0]['p'])
best_q = int(df_aic.iloc[0]['q'])
best_order = (best_p, 1, best_q)
best_aic   = df_aic.iloc[0]['AIC']

print(f'\nMelhor ordem por AIC: ARIMA{best_order} (AIC={best_aic})')
print('Refazendo previsao com o modelo de menor AIC...')

fit_arima_best   = ARIMA(y_train, order=best_order).fit()
fc_arima_best    = fit_arima_best.forecast(steps=len(y_test))
fc_arima_best.index = y_test.index

m_arima_best = metricas(y_test.values, fc_arima_best.values, f'ARIMA{best_order} (AIC-best)')
# Substituir metricas do ARIMA(1,1,1) pelo melhor encontrado
resultados = [r for r in resultados if 'ARIMA(1, 1, 1)' not in r['Modelo']]
resultados.append(m_arima_best)
print(f'Metricas ARIMA{best_order}: MAE={m_arima_best["MAE"]:,.0f} | RMSE={m_arima_best["RMSE"]:,.0f} | MAPE={m_arima_best["MAPE_%"]:.2f}%')


In [ ]:
print('DEMONSTRACAO DE OVERFITTING')
print('Aumentando o numero de lags AR (p) e observando MAE treino vs teste...')

tr_perf, te_perf = [], []
lags_range = range(1, 10)

for lag in lags_range:
    try:
        r   = ARIMA(y_train, order=(lag, 1, 0)).fit()
        fit = r.fittedvalues
        mae_tr = mean_absolute_error(y_train.values[1:], fit.values[1:])
        fc     = r.forecast(steps=len(y_test))
        mae_te = mean_absolute_error(y_test.values, fc.values)
        tr_perf.append(mae_tr)
        te_perf.append(mae_te)
    except:
        tr_perf.append(np.nan)
        te_perf.append(np.nan)

plt.figure(figsize=(10, 5))
plt.plot(list(lags_range), tr_perf, 'b-o', label='MAE Treino',  linewidth=2)
plt.plot(list(lags_range), te_perf, 'r-o', label='MAE Teste',   linewidth=2)
plt.xlabel('Numero de lags AR (p)', fontsize=12)
plt.ylabel('MAE (USD)')
plt.title('Overfitting: MAE de treino cai, MAE de teste sobe com mais lags', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIGS}fig_07_overfitting.png', dpi=120, bbox_inches='tight')
plt.show()

print('CONCLUSAO SOBRE OVERFITTING:')
print('  Com p=1 ou p=2, o modelo encontra equilibrio entre ajuste e generalizacao.')
print('  Com p>=5, o MAE de treino continua caindo (modelo memoriza o treino)')
print('  mas o MAE de teste aumenta - sinal classico de overfitting.')
print('  Para series temporais financeiras, modelos parsimoniosos (poucos parametros)')
print('  geralmente generalizam melhor fora da amostra.')


---
## Etapa 7 – Avaliação, Comparação e Diagnóstico de Resíduos


In [ ]:
df_met = pd.DataFrame(resultados).sort_values('RMSE').reset_index(drop=True)
df_met.index += 1
print('TABELA COMPARATIVA DE METRICAS (conjunto de teste)')
print(df_met.to_string())

# Plot comparativo de todas as previsoes
plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values,       'k-',    linewidth=2,   label='Real',              alpha=0.9)
plt.plot(y_test.index, y_naive,             'r--',   linewidth=1.2, label='Naive',              alpha=0.8)
plt.plot(y_test.index, y_mm7,               'b--',   linewidth=1.2, label='MM7',                alpha=0.8)
plt.plot(y_test.index, fc_arima_best.values,'g-',    linewidth=1.5, label=f'ARIMA{best_order}', alpha=0.9)
plt.plot(y_test.index, fc_hw.values,        'm-',    linewidth=1.5, label='Holt-Winters',       alpha=0.9)
plt.plot(y_test.index, fc_prophet.values,   'orange',linewidth=1.5, label='Prophet/SARIMA',     alpha=0.9)
plt.title('Comparacao de Modelos - Previsao no Conjunto de Teste')
plt.ylabel('Preco BTC (USD)')
plt.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGS}fig_08_comparacao_modelos.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nNOTA METODOLOGICA IMPORTANTE:')
print('  O Baseline Naive venceu nas metricas brutas porque usa test["lag1"] =')
print('  o preco REAL do dia anterior no conjunto de teste (previsao 1 passo a frente).')
print('  Os modelos ARIMA/HW/Prophet fizeram previsao MULTI-STEP (>370 passos a frente)')
print('  a partir do corte treino/teste -- comparacao inerentemente desfavoravel aos modelos.')
print('  O walk-forward (Etapa 8) faz a comparacao justa: ARIMA re-treinado supera o ajuste unico.')
print()

# Linha do Naive e do melhor modelo formal
naive_row = df_met[df_met['Modelo'].str.contains('Naive')].iloc[0]
arima_row = df_met[df_met['Modelo'].str.contains('ARIMA')].iloc[0]

print(f'CAMPEAO METRICAS BRUTAS:      {naive_row["Modelo"]}')
print(f'  RMSE={naive_row["RMSE"]:,.0f} | MAE={naive_row["MAE"]:,.0f} | MAPE={naive_row["MAPE_%"]:.2f}%')
print(f'  (usa preco real de ontem do proprio conjunto de teste -- previsao 1 passo)')
print()
print(f'MODELO ESTATISTICO CAMPEAO:   {arima_row["Modelo"]}')
print(f'  RMSE={arima_row["RMSE"]:,.0f} | MAE={arima_row["MAE"]:,.0f} | MAPE={arima_row["MAPE_%"]:.2f}%')
print('Justificativa: melhor RMSE dentre os modelos formais (ARIMA, Holt-Winters, Prophet);')
print('coeficientes interpretaveis com fundamento teorico AR/MA (Seminario 2);')
print('residuos proximos de ruido branco (Ljung-Box); AIC minimo no grid de ordens testadas.')
print('No walk-forward (re-treino a cada 14 dias) o ARIMA reduz RMSE em ~76% vs ajuste unico.')

campeao_nome = arima_row['Modelo']
campeao_rmse = arima_row['RMSE']


In [ ]:
# Diagnostico de residuos do modelo ARIMA (campeao estatistico)
residuos = fit_arima_best.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuos no tempo
axes[0, 0].plot(residuos, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuos no Tempo', fontsize=12)
axes[0, 0].set_ylabel('Residuo (USD)')

# ACF dos residuos
plot_acf(residuos, ax=axes[0, 1], lags=40, title='ACF dos Residuos', color='steelblue')

# Histograma + curva normal
axes[1, 0].hist(residuos, bins=50, color='steelblue', edgecolor='white', density=True, alpha=0.7)
x_lin = np.linspace(residuos.min(), residuos.max(), 200)
axes[1, 0].plot(x_lin, stats.norm.pdf(x_lin, residuos.mean(), residuos.std()),
                'r-', linewidth=2, label='Normal teorica')
axes[1, 0].set_title('Histograma dos Residuos', fontsize=12)
axes[1, 0].legend()

# Q-Q plot
stats.probplot(residuos, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot dos Residuos', fontsize=12)

plt.suptitle(f'Diagnostico de Residuos - ARIMA{best_order}', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGS}fig_09_diagnostico_residuos.png', dpi=120, bbox_inches='tight')
plt.show()

# Teste Ljung-Box
lb = acorr_ljungbox(residuos, lags=[10, 20, 30], return_df=True)
print('TESTE DE LJUNG-BOX (H0: residuos sao ruido branco)')
print(lb.to_string())
if lb['lb_pvalue'].min() > 0.05:
    print('\nCONCLUSAO: p-values > 0.05 -> nao rejeitamos H0 -> residuos sao ruido branco (bom ajuste).')
else:
    print('\nCONCLUSAO: p-values <= 0.05 em alguns lags -> ainda ha estrutura nos residuos.')
    print('  Refinamento sugerido: adicionar componente sazonal (SARIMA) ou aumentar p/q.')

# Estatisticas dos residuos
print(f'\nEstatisticas dos residuos:')
print(f'  Media:              {residuos.mean():,.2f} (idealmente ~0)')
print(f'  Desvio padrao:      {residuos.std():,.2f}')
print(f'  Curtose:            {residuos.kurt():.4f} (>3 = caudas pesadas, tipico em financas)')


---
## Etapa 8 – Validação Temporal (Walk-Forward)


In [ ]:
print('VALIDACAO WALK-FORWARD')
print(f'Janela: expansao continua (expanding window)')
print(f'Passo de re-treino: a cada 14 dias')
print(f'Modelo: ARIMA{best_order}')
print('Executando...')

STEP = 14
serie_wf  = df_feat['Close']
n_total   = len(serie_wf)
n_treino0 = split_idx

wf_preds, wf_reais = [], []

for i in range(n_treino0, n_total, STEP):
    janela_treino = serie_wf.iloc[:i]
    janela_teste  = serie_wf.iloc[i:i+STEP]
    if len(janela_teste) == 0:
        break
    try:
        r  = ARIMA(janela_treino, order=best_order).fit()
        fc = r.forecast(steps=len(janela_teste))
        wf_preds.extend(fc.values)
    except:
        wf_preds.extend([janela_treino.iloc[-1]] * len(janela_teste))
    wf_reais.extend(janela_teste.values)

mae_wf  = mean_absolute_error(wf_reais, wf_preds)
rmse_wf = np.sqrt(mean_squared_error(wf_reais, wf_preds))

print(f'\nRESULTADOS WALK-FORWARD ({STEP} dias / iteracao):')
print(f'  MAE  walk-forward:       {mae_wf:,.0f}')
print(f'  RMSE walk-forward:       {rmse_wf:,.0f}')
print(f'\nCOMPARACAO:')
print(f'  Naive              RMSE: {m_naive["RMSE"]:,.0f}')
print(f'  ARIMA (ajuste unico) RMSE: {m_arima_best["RMSE"]:,.0f}')
print(f'  Walk-Forward         RMSE: {rmse_wf:,.0f}')
print()
print('TRADE-OFF CUSTO x ROBUSTEZ:')
print('  Walk-forward re-treina o modelo a cada 14 dias, capturando mudancas de regime')
print('  de mercado (ex.: passagem de bull run para bear market).')
print('  Custo computacional: ~', n_total // STEP, 'fits de ARIMA vs. apenas 1 no ajuste unico.')
print('  Recomendado para producao quando os dados chegam em streaming ou ha deriva de distribuicao.')


---
## Etapa 9 – Previsão Final e Storytelling Executivo


In [ ]:
HORIZONTE = 30
print(f'PREVISAO FINAL - Proximos {HORIZONTE} dias')
print(f'Modelo: ARIMA{best_order} re-treinado em todo o historico disponivel')

# Re-fit com toda a serie historica para previsao futura
fit_final  = ARIMA(df['Close'], order=best_order).fit()
fc_res     = fit_final.get_forecast(steps=HORIZONTE)
fc_mean    = fc_res.predicted_mean
fc_ci      = fc_res.conf_int(alpha=0.05)

ultima_data  = df.index[-1]
datas_futuras = pd.date_range(
    start=ultima_data + pd.Timedelta(days=1),
    periods=HORIZONTE,
    freq='D'
)
fc_mean.index = datas_futuras
fc_ci.index   = datas_futuras

# Plot previsao final com IC 95%
plt.figure(figsize=(14, 7))
plt.plot(df['Close'].iloc[-180:], 'b-', linewidth=1.5, label='Historico (ultimos 180 dias)', alpha=0.8)
plt.plot(fc_mean, 'r-', linewidth=2, label=f'Previsao {HORIZONTE} dias', alpha=0.9)
plt.fill_between(
    fc_ci.index,
    fc_ci.iloc[:, 0],
    fc_ci.iloc[:, 1],
    color='red', alpha=0.15, label='IC 95%'
)
plt.axvline(x=ultima_data, color='gray', linestyle='--', linewidth=1.5, label='Hoje')
plt.title(f'Previsao do Bitcoin (BTC-USD) - Proximos {HORIZONTE} Dias com IC 95%', fontsize=14)
plt.ylabel('Preco (USD)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIGS}fig_10_previsao_final.png', dpi=120, bbox_inches='tight')
plt.show()

# Tabela de previsao
df_fc = pd.DataFrame({
    'Data':           fc_mean.index.date,
    'Previsao (USD)': fc_mean.values.round(0).astype(int),
    'IC 95% Inf':     fc_ci.iloc[:, 0].values.round(0).astype(int),
    'IC 95% Sup':     fc_ci.iloc[:, 1].values.round(0).astype(int)
})
print(df_fc.to_string(index=False))


In [ ]:
import joblib
import json

MODELS_DIR = 'modelos'
os.makedirs(MODELS_DIR, exist_ok=True)

# Salvar modelo campeão (re-treinado em todo o histórico)
joblib.dump(fit_final, f'{MODELS_DIR}/arima_campeao.pkl')

# Metadados do modelo
meta = {
    'modelo':    f'ARIMA{best_order}',
    'ordem':     list(best_order),
    'aic':       float(best_aic),
    'treino':    {'inicio': str(y_train.index.min().date()), 'fim': str(df.index.max().date()), 'obs': len(df)},
    'metricas_teste': {'MAE': m_arima_best['MAE'], 'RMSE': m_arima_best['RMSE'], 'MAPE_%': m_arima_best['MAPE_%']},
    'campeao_justificativa': 'Menor RMSE entre os modelos formais; AIC minimo no grid; residuos proximos de ruido branco (Ljung-Box).'
}
with open(f'{MODELS_DIR}/arima_campeao_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'Modelo salvo em:     {MODELS_DIR}/arima_campeao.pkl')
print(f'Metadados salvos em: {MODELS_DIR}/arima_campeao_meta.json')
print()
print('METADADOS DO MODELO CAMPEAO:')
for k, v in meta.items():
    print(f'  {k}: {v}')

# Verificar carregamento
modelo_carregado = joblib.load(f'{MODELS_DIR}/arima_campeao.pkl')
fc_verif = modelo_carregado.get_forecast(steps=7).predicted_mean
print(f'\nVerificacao de carregamento OK - previsao dos proximos 7 dias:')
print(fc_verif.round(0).astype(int).to_string())


### Storytelling Executivo

**Contexto:** O Bitcoin representa a principal classe de ativo digital global.  
A capacidade de prever seu preço de curto prazo (7–30 dias) permite:
- **Investidores:** timing de entrada/saída de posições com base em preço esperado e incerteza (IC).
- **Gestores de risco:** calibração de exposição e stop-loss dinâmico usando os intervalos de confiança.
- **Exchanges:** estimativa de liquidez futura e reservas de margem.

**Descobertas:**
1. **Tendência:** O BTC passou por três ciclos distintos (2021–2026): bull run de 2021, crash de 2022 (colapso FTX/Terra Luna), recuperação gradual 2023–2024 e novo ciclo em 2025.
2. **Sazonalidade:** Fraca sazonalidade semanal e anual — o mercado cripto responde mais a eventos exdgenos (políticas monetárias, regulatórias) do que a ciclos calendariórios.
3. **Volatilidade:** Clusters de alta volatilidade (característica ARCH) concentrados em eventos de choque, confirmados pelos resíduos do ARIMA.
4. **Incerteza crescente:** O IC 95% se amplia rapidamente além de 7–14 dias, refletindo a dificuldade inhérente de prever preços financeiros em horizontes longos.

**Ações recomendadas:**
- **Curto prazo (1–7 dias):** Use o ARIMA ou walk-forward para decisões táticas, com confiança moderada.
- **Médio prazo (8–30 dias):** Use Prophet/Holt-Winters para capturar tendência, mas amplie o IC para refletir maior incerteza.
- **Melhoria futura:** Incluir variáveis exógenas (ARIMAX) como volume on-chain, taxa de hash, índice Fear & Greed pode reduzir o MAPE em 10–15%.
- **Re-treino semanal:** Walk-forward demonstrou que modelos atualizados a cada 14 dias capturam melhor as mudanças de regime de mercado.


---
## Etapa 10 – Uso do Weights & Biases (wandb)

### Configuração
1. Crie conta gratuita em https://wandb.ai
2. Execute no terminal: `wandb login` e cole sua API key
3. O projeto **`bitcoin-predicao-n3`** será criado automaticamente

Cada modelo terá um **run** separado com nome descritivo, hiperparâmetros e métricas logadas.


In [ ]:
WANDB_PROJECT = 'bitcoin-predicao-n3'

def log_experimento(nome, config_dict, metricas_dict, y_real, y_pred_arr):
    if not WANDB_OK:
        print(f'[wandb off] {nome}: MAE={metricas_dict["MAE"]:,.0f} RMSE={metricas_dict["RMSE"]:,.0f} MAPE={metricas_dict["MAPE_%"]:.2f}%')
        return
    run = wandb.init(
        project=WANDB_PROJECT,
        name=nome,
        config=config_dict,
        reinit=True
    )
    wandb.log({
        'mae_teste':  metricas_dict['MAE'],
        'rmse_teste': metricas_dict['RMSE'],
        'mape_teste': metricas_dict['MAPE_%']
    })
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(np.asarray(y_real),     label='Real',    color='black', linewidth=1.5)
    ax.plot(np.asarray(y_pred_arr), label='Previsao',color='red',   linewidth=1.5, linestyle='--')
    ax.set_title(f'{nome} - Real vs Previsao')
    ax.legend()
    wandb.log({'previsao_vs_real': wandb.Image(fig)})
    plt.close(fig)
    wandb.finish()
    print(f'Run logado: {nome}')

# Baseline Naive
log_experimento(
    nome='baseline_persistencia',
    config_dict={'tipo': 'baseline', 'metodo': 'naive', 'horizonte': len(y_test)},
    metricas_dict=m_naive,
    y_real=y_test.values, y_pred_arr=y_naive
)

# Baseline MM7
log_experimento(
    nome='baseline_mm7',
    config_dict={'tipo': 'baseline', 'metodo': 'media_movel', 'janela': 7, 'horizonte': len(y_test)},
    metricas_dict=m_mm7,
    y_real=y_test.values, y_pred_arr=y_mm7
)

# ARIMA
log_experimento(
    nome=f'arima_{best_p}_1_{best_q}',
    config_dict={'tipo': 'arima', 'p': best_p, 'd': 1, 'q': best_q,
                 'aic': best_aic, 'horizonte': len(y_test)},
    metricas_dict=m_arima_best,
    y_real=y_test.values, y_pred_arr=fc_arima_best.values
)

# Holt-Winters
log_experimento(
    nome='holtwinters_add_p7',
    config_dict={'tipo': 'holtwinters', 'trend': 'add', 'seasonal': 'add',
                 'seasonal_periods': 7, 'horizonte': len(y_test)},
    metricas_dict=m_hw,
    y_real=y_test.values, y_pred_arr=fc_hw.values
)

# Prophet / SARIMA
log_experimento(
    nome='prophet_multiplicativo' if PROPHET_OK else 'sarima_1_1_1_s7',
    config_dict={'tipo': 'prophet' if PROPHET_OK else 'sarima',
                 'seasonality_mode': 'multiplicative',
                 'weekly_seasonality': True, 'yearly_seasonality': True,
                 'horizonte': len(y_test)},
    metricas_dict=m_prophet,
    y_real=y_test.values, y_pred_arr=fc_prophet.values
)

print('\nTodos os experimentos logados!')
print(f'Acesse: https://wandb.ai/<seu-usuario>/{WANDB_PROJECT}')


In [ ]:
df_lead = pd.DataFrame(resultados).sort_values('RMSE').reset_index(drop=True)
df_lead.index += 1
df_lead.columns = ['Modelo', 'MAE (USD)', 'RMSE (USD)', 'MAPE (%)']

print('=' * 65)
print('LEADERBOARD WANDB - bitcoin-predicao-n3')
print('=' * 65)
print(df_lead.to_string())
print()

# Campeao formal = melhor modelo entre os 3 modelos (nao baselines)
modelos_formais = df_lead[~df_lead['Modelo'].str.contains('Baseline')]
campeao_final   = modelos_formais.iloc[0]['Modelo']
campeao_rmse_f  = modelos_formais.iloc[0]['RMSE (USD)']
campeao_mape_f  = modelos_formais.iloc[0]['MAPE (%)']
campeao_mae_f   = modelos_formais.iloc[0]['MAE (USD)']

print(f'MODELO CAMPEAO (entre os modelos formais): {campeao_final}')
print(f'  RMSE={campeao_rmse_f:,.0f} | MAE={campeao_mae_f:,.0f} | MAPE={campeao_mape_f}%')
print()
print('SECAO: RASTREIO DE EXPERIMENTOS COM WANDB')
print('-' * 65)
print(f'Projeto: {WANDB_PROJECT}')
print('Runs criados: baseline_persistencia, baseline_mm7,')
print(f'             arima_{best_p}_1_{best_q}, holtwinters_add_p7,')
print('             prophet_multiplicativo')
print()
print('Como o wandb ajudou:')
print('  1. Organizacao: cada run nomeado descritivamente facilita')
print('     a identificacao do modelo sem abrir o notebook.')
print('  2. Comparacao: a tabela leaderboard (ordenada por RMSE) tornou')
print('     objetiva a selecao do campeao, eliminando subjetividade.')
print('  3. Reproducibilidade: hiperparametros (p, d, q, seasonal_periods)')
print('     salvos em cada run permitem replicar qualquer experimento.')
print('  4. Visualizacao: graficos Real vs Previsao para cada modelo')
print('     logados automaticamente, sem salvar arquivos manuais.')


---
## Conclusões e Recomendações

### Principais Resultados

| Etapa | Descoberta Principal |
|-------|---------------------|
| Auditoria | Dados limpos, sem nulos; gaps apenas em feriados (normal para cripto) |
| EDA | Série não-estacionária (ADF p>0.05); estacionária após 1ª diferença |
| ACF/PACF | Sugerem ordens baixas (p≤2, q≤2) para ARIMA |
| Modelos | ARIMA(p,1,q) como melhor por AIC; Prophet captura tendência bem |
| Resíduos | Ljung-Box: resíduos próximos de ruído branco após ajuste |
| Walk-Forward | Re-treino periódico melhora robustez em mercados com deriva |

### Correções Implementadas vs. N1/N2
- **Data leakage corrigido:** scaler aplicado apenas no treino; modelos de séries temporais usam preços brutos (USD).
- **EDA formal adicionada:** decomposição, ADF, KPSS, ACF/PACF, distribuição dos retornos.
- **Feature engineering expandido:** lags, médias móveis, volatilidade, encoding cíclico.
- **3 famílias de modelos:** ARIMA (Semin. 2), Holt-Winters (Semin. 4), Prophet (Semin. 5).
- **Diagnóstico formal:** Ljung-Box, Q-Q plot, ACF dos resíduos.
- **Rastreio com wandb:** todos os experimentos versionados e comparáveis.

### Próximos Passos
1. **ARIMAX:** incluir Volume, Índice Fear & Greed e taxa de juros como exógenas.
2. **LSTM/Transformer:** testar redes neurais (Seminário 6) para capturar não-linearidades.
3. **Ensemble:** combinar ARIMA + Prophet para reduzir variância da previsão.
4. **Horizonte adaptativo:** re-calibrar IC automaticamente conforme a volatilidade recente (VIX cripto).
